# CICIDS2017 Intrusion Detection

A streamlined multiclass workflow: load the CSV files, remove exact duplicate rows, clean invalid values, split the data, undersample BENIGN training rows, use SMOTE to increase minority classes, compare four models with macro F1, and save the best model.


## 1. Imports


In [1]:
import gc
from pathlib import Path

import numpy as np
import pandas as pd
import joblib
import lightgbm as lgb
import xgboost as xgb
from imblearn.over_sampling import SMOTE
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, f1_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler

RANDOM_STATE = 42


## 2. Load and deduplicate the data


In [2]:
project_path = Path(r'C:\Users\Fahad\Desktop\letters_of_recommendation')
base_path = project_path / 'CIC_IDS2017'
csv_files = sorted(base_path.rglob('*.csv'))
if not csv_files:
    raise FileNotFoundError(f'No CSV files found under {base_path}')

frames = []
for file in csv_files:
    frame = pd.read_csv(file, low_memory=False)
    frame.columns = frame.columns.str.strip()
    frame = frame.loc[:, ~frame.columns.duplicated()]
    # Downcast numeric columns to reduce RAM use while combining the CSVs.
    for col in frame.select_dtypes(include=['integer']).columns:
        frame[col] = pd.to_numeric(frame[col], downcast='integer')
    for col in frame.select_dtypes(include=['floating']).columns:
        frame[col] = pd.to_numeric(frame[col], downcast='float')
    frames.append(frame)
    print(f'Loaded {file.name}: {len(frame):,} rows')

df = pd.concat(frames, ignore_index=True)
del frames
rows_before_deduplication = len(df)
df = df.drop_duplicates(keep='first').reset_index(drop=True)
print(f'Exact duplicate rows removed: {rows_before_deduplication - len(df):,}')
print(f'Rows remaining: {len(df):,}')


Loaded Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv: 225,745 rows
Loaded Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv: 286,467 rows
Loaded Friday-WorkingHours-Morning.pcap_ISCX.csv: 191,033 rows
Loaded Monday-WorkingHours.pcap_ISCX.csv: 529,918 rows
Loaded Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv: 288,602 rows
Loaded Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv: 170,366 rows
Loaded Tuesday-WorkingHours.pcap_ISCX.csv: 445,909 rows
Loaded Wednesday-workingHours.pcap_ISCX.csv: 692,703 rows
Exact duplicate rows removed: 308,387
Rows remaining: 2,522,356


## 3. Clean, split, and scale

The split happens before SMOTE so synthetic samples are made from training data only. Scaling is needed before SMOTE because it uses nearest neighbors; the same scaler is applied to the test set.


In [3]:
label_col = next((col for col in df.columns if col.lower() == 'label'), None)
if label_col is None:
    raise ValueError('Could not find the Label column.')

# Keep the class label separate. Drop any precomputed attack flag to avoid leakage.
y = df[label_col]
X = df.drop(columns=[label_col, 'is_attack'], errors='ignore')
X = X.apply(pd.to_numeric, errors='coerce')
X = X.replace([np.inf, -np.inf], np.nan)

valid_rows = y.notna() & X.notna().all(axis=1)
X = X.loc[valid_rows]
y = y.loc[valid_rows].astype(str).str.strip()
nonblank_labels = y.ne('')
X = X.loc[nonblank_labels].astype(np.float32)
y = y.loc[nonblank_labels]

le = LabelEncoder()
y_encoded = le.fit_transform(y)
feature_columns = X.columns.tolist()

X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y_encoded,
)
print(f'Training rows before balancing: {len(X_train):,}')
print(f'Test rows (left untouched): {len(X_test):,}')
print(f'Features: {len(feature_columns)} | Classes: {len(le.classes_)}')

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train).astype(np.float32)
X_test_scaled = scaler.transform(X_test).astype(np.float32)

del df, X, y, X_train, X_test, y_encoded
gc.collect()


Training rows before balancing: 2,016,633
Test rows (left untouched): 504,159
Features: 78 | Classes: 15


16

## 4. Undersample and use SMOTE

To keep the training set manageable locally, retain 15% of the BENIGN training rows. Then use SMOTE one attack class at a time, increasing each class below the target to half the retained BENIGN count. This limits peak memory. Classes with fewer than two examples cannot be synthesized, so they stay at their observed count. The test set is never resampled.


In [4]:
benign_matches = [i for i, name in enumerate(le.classes_) if name.strip().upper() == 'BENIGN']
if not benign_matches:
    raise ValueError("Could not find the 'BENIGN' class required for undersampling.")
benign_class = benign_matches[0]

rng = np.random.default_rng(RANDOM_STATE)
benign_indices = np.flatnonzero(y_train == benign_class)
benign_keep_count = max(1, int(len(benign_indices) * 0.15))
benign_keep_indices = rng.choice(benign_indices, size=benign_keep_count, replace=False)
target_size = max(2, int(benign_keep_count * 0.5))

X_parts = [X_train_scaled[benign_keep_indices]]
y_parts = [np.full(benign_keep_count, benign_class, dtype=y_train.dtype)]
print(f'BENIGN rows kept: {benign_keep_count:,} of {len(benign_indices):,}')
print(f'Target for smaller attack classes: {target_size:,}')

for class_id in np.unique(y_train):
    if class_id == benign_class:
        continue

    class_indices = np.flatnonzero(y_train == class_id)
    X_class = X_train_scaled[class_indices]
    y_class = y_train[class_indices]
    class_count = len(class_indices)

    if class_count >= target_size or class_count < 2:
        # Keep classes already at target, and classes too small for SMOTE.
        X_parts.append(X_class)
        y_parts.append(y_class)
        continue

    # A small BENIGN reference sample keeps each SMOTE operation memory-light.
    benign_reference_indices = benign_keep_indices[:min(100, benign_keep_count)]
    X_reference = X_train_scaled[benign_reference_indices]
    y_reference = np.full(len(benign_reference_indices), benign_class, dtype=y_train.dtype)
    X_for_smote = np.concatenate([X_reference, X_class], axis=0)
    y_for_smote = np.concatenate([y_reference, y_class])

    neighbors = min(5, class_count - 1)
    sampler = SMOTE(
        sampling_strategy={benign_class: len(benign_reference_indices), int(class_id): target_size},
        k_neighbors=neighbors,
        random_state=RANDOM_STATE,
    )
    X_resampled, y_resampled = sampler.fit_resample(X_for_smote, y_for_smote)
    attack_rows = y_resampled == class_id
    X_parts.append(np.asarray(X_resampled[attack_rows], dtype=np.float32))
    y_parts.append(y_resampled[attack_rows])

    del X_class, y_class, X_reference, y_reference, X_for_smote, y_for_smote
    del X_resampled, y_resampled, attack_rows
    gc.collect()

X_train_balanced = np.concatenate(X_parts, axis=0)
y_train_balanced = np.concatenate(y_parts)
del X_parts, y_parts, X_train_scaled, y_train

gc.collect()
print(f'Balanced training rows: {len(y_train_balanced):,}')


BENIGN rows kept: 251,406 of 1,676,040
Target for smaller attack classes: 125,703
Balanced training rows: 2,023,822


## 5. Train and compare four models

Macro F1 gives each attack class equal weight, which is useful when the classes are imbalanced.


In [5]:
n_classes = len(le.classes_)
model_builders = {
    'Logistic Regression': lambda: LogisticRegression(max_iter=200, random_state=RANDOM_STATE),
    'Random Forest': lambda: RandomForestClassifier(
        n_estimators=100, max_depth=20, random_state=RANDOM_STATE, n_jobs=-1,
    ),
    'XGBoost': lambda: xgb.XGBClassifier(
        n_estimators=100, max_depth=8, learning_rate=0.1,
        objective='multi:softprob', num_class=n_classes,
        eval_metric='mlogloss', tree_method='hist',
        random_state=RANDOM_STATE, n_jobs=-1,
    ),
    'LightGBM': lambda: lgb.LGBMClassifier(
        n_estimators=100, learning_rate=0.1,
        objective='multiclass', num_class=n_classes,
        random_state=RANDOM_STATE, verbosity=-1, n_jobs=-1,
    ),
}

results = []
best_name = None
best_model = None
best_score = -1

for name, build_model in model_builders.items():
    print(f'Training {name}...')
    model = build_model()
    model.fit(X_train_balanced, y_train_balanced)
    predictions = model.predict(X_test_scaled)
    score = f1_score(y_test, predictions, average='macro', zero_division=0)
    results.append({'Model': name, 'Macro F1': score})
    print(f'  Macro F1: {score:.4f}')
    if score > best_score:
        best_name, best_model, best_score = name, model, score
    del model, predictions
    gc.collect()

del X_train_balanced, y_train_balanced
gc.collect()

results_df = pd.DataFrame(results).sort_values('Macro F1', ascending=False)
print('\nModel comparison:')
print(results_df.to_string(index=False))


Training Logistic Regression...


c:\Users\Fahad\Desktop\letters_of_recommendation\.env\Lib\site-packages\sklearn\linear_model\_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 200 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=200).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


  Macro F1: 0.5108
Training Random Forest...
  Macro F1: 0.8129
Training XGBoost...
  Macro F1: 0.8802
Training LightGBM...
  Macro F1: 0.1689

Model comparison:
              Model  Macro F1
            XGBoost  0.880205
      Random Forest  0.812855
Logistic Regression  0.510844
           LightGBM  0.168876


## 6. Review and save the best model


In [6]:
best_predictions = best_model.predict(X_test_scaled)

print(f'Best model by macro F1: {best_name} ({best_score:.4f})')
print(classification_report(
    y_test,
    best_predictions,
    labels=np.arange(len(le.classes_)),
    target_names=le.classes_,
    zero_division=0,
))

models_path = project_path / 'models'
models_path.mkdir(parents=True, exist_ok=True)
model_path = models_path / 'best_intrusion_detection_model.joblib'

joblib.dump(
    {
        'model': best_model,
        'label_encoder': le,
        'scaler': scaler,
        'feature_columns': feature_columns,
        'model_name': best_name,
    },
    model_path,
)
print(f'Saved the best model to {model_path}')


Best model by macro F1: XGBoost (0.8802)
                            precision    recall  f1-score   support

                    BENIGN       1.00      1.00      1.00    419011
                       Bot       0.40      0.99      0.57       390
                      DDoS       1.00      1.00      1.00     25603
             DoS GoldenEye       0.99      1.00      0.99      2057
                  DoS Hulk       1.00      1.00      1.00     34569
          DoS Slowhttptest       0.97      1.00      0.98      1046
             DoS slowloris       1.00      0.99      0.99      1077
               FTP-Patator       0.99      1.00      1.00      1186
                Heartbleed       1.00      1.00      1.00         2
              Infiltration       0.88      1.00      0.93         7
                  PortScan       0.99      1.00      0.99     18139
               SSH-Patator       1.00      1.00      1.00       644
  Web Attack � Brute Force       0.80      0.70      0.75       294
Web At